In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('3_1').getOrCreate()

In [2]:
customers_df = spark.read \
                .option('header', True) \
                .option('inferSchema', True) \
                .csv('customers.csv')

transactions_df = spark.read \
                    .option('header', True) \
                    .option('inferSchema', True) \
                    .csv('transactions.csv')

In [3]:
# Customers --> lower(status), dropDuplicates(['cust_id', 'name', 'status']
# Transactions --> cust_id in customers, eur to usd & null = 0, 
    # 30 day rolling total (window) for each customer by timestamp, rank (row_number) customers by their lifettime spend

In [21]:
from pyspark.sql.functions import col, lower, to_date, when
customers_filtered = customers_df.withColumn('status', lower(col('status'))) \
                        .withColumn('signup_date', to_date('signup_date')) \
                        .filter(col('status') == 'active') \
                        .withColumn('signup_date', when(col('signup_date').isNull(), '1999-01-01').otherwise(col('signup_date'))) \
                        .dropDuplicates(['cust_id', 'name', 'status'])

customers_filtered.show()

+-------+-------+-----------+------+
|cust_id|   name|signup_date|status|
+-------+-------+-----------+------+
|      1|  Alice| 2023-01-01|active|
|      3|Charlie| 1999-01-01|active|
|      4|   Dave| 2023-03-01|active|
+-------+-------+-----------+------+



In [34]:
from pyspark.sql.functions import when, round, lit, unix_timestamp, row_number, desc
from pyspark.sql import Window

window_spec = (
    Window.partitionBy('tx_id').orderBy(desc('tx_timestamp'))
)
transaction_df_filtered = transactions_df.withColumn(
                                'amount', when(
                                    col('currency') == 'EUR', round(col('amount')*1.10,2)
                                ).otherwise(col('amount'))
                            ) \
                            .withColumn('currency', when(col('currency') == 'EUR', lit('USD')).otherwise(col('currency'))) \
                            .withColumn('amount', col('amount').cast('float')) \
                            .fillna(0, subset = ['amount']) \
                            .withColumn('row_number', row_number().over(window_spec)).filter(col('row_number')==1)
                            

transaction_df_filtered.show()

+-----+-------+------+--------+-------------------+----------+
|tx_id|cust_id|amount|currency|       tx_timestamp|row_number|
+-----+-------+------+--------+-------------------+----------+
|  101|      1|  20.0|     USD|2023-03-01 10:10:00|         1|
|  102|      1|  75.5|     USD|2023-03-01 10:00:00|         1|
|  103|      2| 110.0|     USD|2023-03-05 12:30:00|         1|
|  104|      1|   0.0|     USD|2023-03-10 14:00:00|         1|
|  105|     99| 200.0|     USD|2023-03-12 09:00:00|         1|
|  106|      4| 500.0|     USD|2023-04-05 11:00:00|         1|
+-----+-------+------+--------+-------------------+----------+



In [35]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, desc, sum

rolling_window = (
    Window.partitionBy('cust_id').orderBy(unix_timestamp(col('tx_timestamp'))).rangeBetween(-30*86400, Window.currentRow)
)

transaction_df_combined = transaction_df_filtered.withColumn('30_day_rolling_total', sum('amount').over(rolling_window)) \
.join(customers_filtered, ['cust_id'], how='right') \
.fillna(0, subset=['amount']) \
.fillna(0, subset=['30_day_rolling_total'])

transaction_df_combined.show()

+-------+-----+------+--------+-------------------+----------+--------------------+-------+-----------+------+
|cust_id|tx_id|amount|currency|       tx_timestamp|row_number|30_day_rolling_total|   name|signup_date|status|
+-------+-----+------+--------+-------------------+----------+--------------------+-------+-----------+------+
|      1|  104|   0.0|     USD|2023-03-10 14:00:00|         1|                95.5|  Alice| 2023-01-01|active|
|      1|  101|  20.0|     USD|2023-03-01 10:10:00|         1|                95.5|  Alice| 2023-01-01|active|
|      1|  102|  75.5|     USD|2023-03-01 10:00:00|         1|                75.5|  Alice| 2023-01-01|active|
|      3| NULL|   0.0|    NULL|               NULL|      NULL|                 0.0|Charlie| 1999-01-01|active|
|      4|  106| 500.0|     USD|2023-04-05 11:00:00|         1|               500.0|   Dave| 2023-03-01|active|
+-------+-----+------+--------+-------------------+----------+--------------------+-------+-----------+------+



In [36]:

lifetime_spend_df = transaction_df_combined.groupBy('cust_id').agg(sum(col('amount')).alias('lifetime_spend')).orderBy('cust_id')

rownum_window = (
    Window.orderBy(desc('lifetime_spend'))
)

lifetime_spend_df.withColumn('row_number', row_number().over(rownum_window)).show()

+-------+--------------+----------+
|cust_id|lifetime_spend|row_number|
+-------+--------------+----------+
|      4|         500.0|         1|
|      1|          95.5|         2|
|      3|           0.0|         3|
+-------+--------------+----------+

